# 👀 Heid Doon — a Gemma 4 study companion that catches you procrastinating
### *It reads your work, not just your screen.* Contract → semantic verdicts → negotiated breaks → drift autopsy → adapted plan.
**Build with Gemma: GDGoC Aberdeen · Track: Motivation & Habits**

Study tools that only answer questions are easy to ignore. Heid Doon intervenes: it judges your screen
**semantically against rules you wrote** (a YouTube *lecture* passes; cats don't), catches phone-drift via
camera frames, and — the deeper trick — **reads real progress from your working file itself**: twenty minutes
on your phone shows up as an empty diff. *You can hide a phone from a camera. You can't hide an empty page.*

**This notebook is the code repo AND the demo.** Every mechanic below is a live Gemma 4 call — Run All
reproduces every verdict on the bundled test set, prints the eval table, then launches the interactive app.
The privacy-critical local watcher (`watcher.py`) is written by this notebook and runs on the student's
laptop via Ollama — frames never leave the machine. **Open weights are why this product can exist at all.**

> ⚙️ Setup: **Accelerator = GPU · Internet = ON** · attach the Gemma 4 model via *Add Input → Models*
> and the test set via *Add Input → Datasets*.

In [ ]:
# ── 0 · Setup ─────────────────────────────────────────────────────────────
!pip install -q -U transformers accelerate gradio pillow

import json, re, os, glob, time, statistics, torch
from PIL import Image
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ── 1 · Load Gemma 4 ──────────────────────────────────────────────────────
# Preferred: attach Gemma 4 via Add Input → Models (no tokens needed) — auto-detected below.
# Fallback: set MODEL_ID to the participant guide's handle.       ▼▼ EDIT IF NEEDED ▼▼
MODEL_ID = ""                       # e.g. "google/gemma-4-e4b-it" per participant guide
MOCK = False                        # becomes True only if loading fails (loud warnings)

if not MODEL_ID:                    # auto-detect an attached Kaggle Model
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)]
    hits = [h for h in hits if "gemma" in h.lower()]
    if hits: MODEL_ID = sorted(hits, key=len)[0]; print("auto-detected model:", MODEL_ID)

pipe = None
try:
    from transformers import pipeline
    dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    pipe = pipeline("image-text-to-text", model=MODEL_ID, device_map="auto", torch_dtype=dtype)
    print("✓ loaded:", MODEL_ID, "| dtype:", dtype)
except Exception as e:
    MOCK = True
    print("!"*70 + f"\n⚠️  MODEL LOAD FAILED → MOCK MODE (canned outputs, NOT real verdicts)\n"
          f"    Fix MODEL_ID / attach the model before the final Save Version.\n    Error: {e}\n" + "!"*70)

def ask_gemma(prompt, image=None, max_new_tokens=300):
    if MOCK:
        return json.dumps({"on_task": False, "seen": "MOCK MODE", "reason": "model not loaded",
                           "nudge": "Load the real model before submitting.", "frame_kind": "screen"})
    content = ([{"type": "image", "image": image}] if image is not None else []) + [{"type": "text", "text": prompt}]
    out = pipe(text=[{"role": "user", "content": content}], max_new_tokens=max_new_tokens)
    return out[0]["generated_text"][-1]["content"]

def ask_json(prompt, image=None, retries=2):
    """Every Heid Doon mechanic is one of these: a Gemma structured-output call."""
    last = ""
    for _ in range(retries):
        last = ask_gemma(prompt + "\nReply with ONLY one JSON object, no prose.", image)
        m = re.search(r"\{.*\}", last, re.S)
        if m:
            try: return json.loads(m.group(0))
            except json.JSONDecodeError: continue
    return {"error": "unparseable", "raw": last[:400]}

## 2 · The Contract — the student writes the rules
Everything downstream is judged **against this** (autonomy by design — monitoring you opted into, with
rules you authored, is a study partner; anything else would be surveillance). Natural language in →
schema out: the function-calling pattern that powers every mechanic here.

In [ ]:
CONTRACT_TEXT = """I'm revising thermodynamics chapter 4 (entropy) until 12:30.
Exam Friday — no all-nighter this time. Lecture videos and PDFs about thermodynamics are fine,
music is fine, my study group chat is fine when we're discussing the problem set.
No social media, no entertainment videos. Track my notes file. Camera presence on. Kind but sharp."""

CONTRACT = ask_json(f"""Compile this study contract into exactly this schema:
{{"task": str, "allowed": [str], "blocked": [str], "artifacts": [str],
  "signals": [str], "tone": str, "ends": str}}
Contract: {CONTRACT_TEXT}""")
print(json.dumps(CONTRACT, indent=2))

## 3 · The Watcher — semantic verdicts (screen *and* camera frames)
The point judges should test us on: **a window-title blocker cannot do this.** Our test set includes the
cases that break blocklists — a YouTube *lecture* (allowed), a PDF for the *wrong module* (drift), a chat
app *discussing the problem set* (allowed). Only semantic understanding of the frame vs the contract gets
these right — that's Gemma 4's multimodal vision doing irreplaceable work.

In [ ]:
VERDICT_PROMPT = """You are a kind but sharp focus watcher for a student. Their contract: {contract}
Look at this frame — it may be a SCREEN capture or a WEBCAM frame. Judge MEANING against the contract:
- video sites are ON task only if the content is course-relevant (a lecture) AND lectures are allowed
- documents/PDFs are ON task only if they match the contracted module
- chat is ON task only if the conversation is about the coursework
- webcam frames: phone-in-hand or an empty chair is OFF task; working at the desk is ON task
Schema: {{"frame_kind": "screen|camera", "on_task": true/false, "seen": "what app/site/content",
"reason": "one line: the judgment", "nudge": "if off task: ONE short warm line, no shame, no exclamation marks"}}"""

def verdict(img):
    t0 = time.time()
    v = ask_json(VERDICT_PROMPT.format(contract=json.dumps(CONTRACT)), image=img)
    v["_latency_s"] = round(time.time() - t0, 1)
    return v

In [ ]:
# ── 4 · THE EVAL — the number for the writeup ────────────────────────────
TEST_DIR = next(iter(glob.glob("/kaggle/input/*heid*doon*")), "/kaggle/input/heid-doon-testset")
labels = json.load(open(f"{TEST_DIR}/labels.json"))

results, lat = [], []
for fname, meta in labels.items():
    path = f"{TEST_DIR}/{fname}"
    if fname.startswith("_TODO") or not os.path.exists(path):
        print("· skipped (add a real capture):", fname); continue
    v = verdict(Image.open(path).convert("RGB"))
    ok = v.get("on_task") == meta["on_task"]
    lat.append(v.get("_latency_s", 0))
    results.append({"file": fname, "case": meta["case"], "kind": meta["kind"], "correct": ok,
                    "expected": meta["on_task"], "got": v.get("on_task"), "seen": v.get("seen"), "reason": v.get("reason")})
    print(("✓" if ok else "✗"), f"[{meta['case']:4}]", fname, "→", v.get("seen"), "|", v.get("reason"))

if results:
    n_ok, n = sum(r["correct"] for r in results), len(results)
    hard = [r for r in results if r["case"] == "hard"]
    h_ok = sum(r["correct"] for r in hard)
    summary = {"accuracy": f"{n_ok}/{n}", "hard_cases": f"{h_ok}/{len(hard)}",
               "median_latency_s": statistics.median(lat) if lat else None, "mock_mode": MOCK}
    json.dump({"summary": summary, "results": results}, open("eval_results.json", "w"), indent=2)
    print("\n★ WRITEUP NUMBERS →", summary)
    if MOCK: print("⚠️  MOCK MODE — these are NOT real numbers. Do not quote them.")

## 5 · Work-diff — progress read from the artifact, not the activity
**The phone answer.** Heid Doon snapshots the contracted file; Gemma judges the *delta* — substance,
padding, or stalled. Device-independent by construction: procrastinate on any device you like, the diff
tells the truth. Below: a real revision delta vs a padding delta.

In [ ]:
DIFF_PROMPT = """Two snapshots of a student's working file, ~20 minutes apart. Contract: {contract}
Judge the DELTA only. Padding (filler, repetition, vague promises to study later) is NOT progress.
Schema: {{"delta_words": int, "substantive": true/false, "summary": "one line",
"quality_note": "one specific observation about the new content, e.g. a gap or a good step",
"verdict": "progress|padding|stalled"}}
--- BEFORE ---
{prev}
--- AFTER ---
{curr}"""

def diff_artifact(prev, curr):
    return ask_json(DIFF_PROMPT.format(contract=json.dumps(CONTRACT), prev=prev[:6000], curr=curr[:6000]))

v1 = open(f"{TEST_DIR}/notes_v1.md").read()
v2 = open(f"{TEST_DIR}/notes_v2.md").read()
v2p = open(f"{TEST_DIR}/notes_v2_padding.md").read()
print("REAL WORK  →", json.dumps(diff_artifact(v1, v2), indent=2))
print("\nPADDING    →", json.dumps(diff_artifact(v1, v2p), indent=2))
print("\nSTALLED    →", json.dumps(diff_artifact(v1, v1), indent=2))

## 6 · The Bouncer — breaks are earned, not stolen
Ask for a break → answer one retrieval question generated **from your own notes** (retrieval practice:
the "cost" of a break is literally learning). Graded kindly but honestly via structured output.

In [ ]:
def bouncer_question(notes):
    return ask_json(f"""From these study notes, write ONE short retrieval question the student should
answer before earning a 10-minute break. Schema: {{"question": str, "key_points": [str]}}
Notes: {notes[:4000]}""")

def grade_answer(q, answer):
    return ask_json(f"""Question: {q.get('question')}   Key points: {q.get('key_points')}
Student answered: "{answer}"
Grade kindly but honestly — partial credit if the core idea is right.
Schema: {{"pass": true/false, "feedback": "one warm line"}}""")

q = bouncer_question(v2); print(json.dumps(q, indent=2))
print(json.dumps(grade_answer(q, "entropy still rises in free expansion because S is a state function so only endpoints matter"), indent=2))

## 7 · Receipt + learner model — the session's honest accounting
The event log (verdicts, diffs, quiz results) becomes a compassionate **drift autopsy** and an updated
**learner model** — weak topics, *drift patterns* (when/where/trigger), best nudge style, next difficulty.
Tomorrow's contract puts the hard material *before* your personal danger zone.

In [ ]:
LEARNER = {"weak_topics": ["entropy calculations"], "strong_topics": [], "drift_patterns": [],
           "avg_focus_streak_min": 0, "best_nudge_style": "dry_humour", "next_difficulty": "same"}

def make_receipt(events, learner):
    return ask_json(f"""Study session event log: {json.dumps(events)}
Current learner model: {json.dumps(learner)}
Produce: 1) a 2-sentence compassionate drift AUTOPSY naming the pattern and its trigger (no shame);
2) the UPDATED learner model (same schema, drift_patterns as short strings);
3) tomorrow's contract line (hard material BEFORE the drift danger zone, one scheduled break);
4) a focus score 0-100.
Schema: {{"autopsy": str, "learner_model": {json.dumps(list(LEARNER.keys()))}, "tomorrow": str, "focus_score": int}}""")

DEMO_EVENTS = [
  {"t": "+02m", "kind": "screen", "on_task": True,  "seen": "thermo notes PDF"},
  {"t": "+11m", "kind": "screen", "on_task": True,  "seen": "YouTube — entropy lecture (allowed)"},
  {"t": "+24m", "kind": "screen", "on_task": False, "seen": "YouTube — cat compilation"},
  {"t": "+26m", "kind": "camera", "on_task": False, "seen": "phone in hand"},
  {"t": "+27m", "kind": "screen", "on_task": True,  "seen": "back on notes"},
  {"t": "+41m", "kind": "quiz",   "pass": False,    "topic": "Carnot cycles"},
  {"t": "+45m", "kind": "diff",   "verdict": "progress", "summary": "+210 words, 2 worked examples"},
]
receipt = make_receipt(DEMO_EVENTS, LEARNER)
print(json.dumps(receipt, indent=2))

## 8 · `watcher.py` — the local, private half
Written into the repo by this cell. Runs on the student's laptop against **Ollama (Gemma 4 E4B)**:
screen frames (mss) + webcam frames (cv2) + input-idle inference, judged locally, **frames discarded,
verdicts only**. Works with Wi-Fi off. This is the half that closed cloud models cannot ethically offer —
and judges can reproduce its exact verdict behaviour in Section 4 above, on the same open weights.

In [ ]:
%%writefile watcher.py
"""Heid Doon local watcher — Gemma 4 E4B via Ollama. Frames NEVER leave the laptop.
Usage:  ollama pull <gemma4-tag>  →  python watcher.py
Deps :  pip install mss opencv-python plyer requests pillow"""
import mss, io, base64, json, time, requests
from PIL import Image
try: import cv2
except ImportError: cv2 = None
try: from plyer import notification
except ImportError: notification = None

OLLAMA, MODEL = "http://localhost:11434/api/generate", "gemma4:e4b"   # tag per participant guide
CONTRACT = json.load(open("contract.json"))
PROMPT = """You are a kind focus watcher. Contract: %s
Frame may be SCREEN or WEBCAM. Judge meaning vs contract (a lecture on a video site can be ON task;
webcam phone-in-hand or empty chair is OFF task). JSON only:
{"on_task": true/false, "seen": "...", "nudge": "one short warm line if off task"}"""

def b64(img):
    img.thumbnail((1024, 640)); buf = io.BytesIO(); img.save(buf, "JPEG", quality=70)
    return base64.b64encode(buf.getvalue()).decode()

def screen_frame():
    with mss.mss() as s:
        shot = s.grab(s.monitors[1]); return Image.frombytes("RGB", shot.size, shot.rgb)

def camera_frame():
    if not cv2: return None
    cap = cv2.VideoCapture(0); ok, f = cap.read(); cap.release()
    return Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) if ok else None

def judge(img):
    r = requests.post(OLLAMA, json={"model": MODEL, "prompt": PROMPT % json.dumps(CONTRACT),
        "images": [b64(img)], "format": "json", "stream": False}, timeout=120)
    return json.loads(r.json()["response"])

if __name__ == "__main__":
    use_cam = "camera" in CONTRACT.get("signals", [])
    i, log = 0, open("events.jsonl", "a")
    print("Heid Doon watching (local only). Ctrl-C to stop.")
    while True:
        img = camera_frame() if (use_cam and i % 2) else screen_frame()
        if img is not None:
            try:
                v = judge(img); v["t"] = time.strftime("%H:%M:%S")
                log.write(json.dumps(v) + "\n"); log.flush()
                print(v["t"], "🟢" if v.get("on_task") else "🟠", v.get("seen"), "|", v.get("nudge", ""))
                if not v.get("on_task") and notification:
                    notification.notify(title="Heid Doon 👀", message=v.get("nudge", "Back to it."), timeout=8)
            except Exception as e:
                print("watch error:", e)
        i += 1; time.sleep(20)   # a rhythm of check-ins, not millisecond policing

In [ ]:
%%writefile contract.json
{"task": "thermodynamics chapter 4 (entropy)",
 "allowed": ["lecture videos", "thermodynamics PDFs and docs", "music", "study-group chat about the problem set"],
 "blocked": ["social media", "entertainment video", "shorts"],
 "artifacts": ["notes_thermo.md"], "signals": ["screen", "camera", "diff"],
 "tone": "kind_but_sharp", "ends": "12:30"}

## 9 · The app — interactive demo (public share link)
Tabs mirror the loop. **Session state is real**: every verdict/diff/quiz you run is logged, and *End
session* generates the receipt from what actually happened. If the share link is unavailable at the venue,
this notebook **is** the demo (an accepted format).

In [ ]:
import gradio as gr

def ui_verdict(img, events):
    if img is None: return "{}", "Upload or snap a frame first.", events
    v = verdict(Image.fromarray(img))
    events = events + [{"t": time.strftime("+%Mm"), "kind": v.get("frame_kind", "screen"),
                        "on_task": v.get("on_task"), "seen": v.get("seen")}]
    banner = "🟢 **ON TASK** — " + str(v.get("reason", "")) if v.get("on_task") \
             else "🟠 **DRIFT** — " + str(v.get("nudge", ""))
    return json.dumps(v, indent=2), banner, events

def ui_diff(prev, curr, events):
    d = diff_artifact(prev or "", curr or "")
    events = events + [{"t": time.strftime("+%Mm"), "kind": "diff", "verdict": d.get("verdict"), "summary": d.get("summary")}]
    return json.dumps(d, indent=2), events

def ui_bouncer_q(notes):
    q = bouncer_question(notes or v2); return json.dumps(q), q.get("question", "")

def ui_bouncer_grade(qjson, answer, events):
    try: q = json.loads(qjson)
    except Exception: return "Generate a question first.", events
    g = grade_answer(q, answer or "")
    events = events + [{"t": time.strftime("+%Mm"), "kind": "quiz", "pass": g.get("pass")}]
    verdict_line = ("✅ Break earned — 10 minutes, I'll come get you. " if g.get("pass")
                    else "❌ Not quite — ") + str(g.get("feedback", ""))
    return verdict_line, events

def ui_receipt(events):
    if not events: return "No events yet — judge a frame or run a diff first.", "{}"
    r = make_receipt(events, LEARNER)
    mdtxt = (f"## Focus score: {r.get('focus_score','?')}/100\n\n**Drift autopsy:** {r.get('autopsy','')}\n\n"
             f"**Tomorrow:** {r.get('tomorrow','')}")
    return mdtxt, json.dumps(r.get("learner_model", {}), indent=2)

with gr.Blocks(title="Heid Doon") as demo:
    gr.Markdown("# 👀 Heid Doon — Gemma 4 study companion\n*It reads your work, not just your screen.*"
                + ("\n\n# ⚠️ MOCK MODE — model not loaded, verdicts are canned" if MOCK else ""))
    events = gr.State([])
    with gr.Tab("1 · Watcher — judge a frame"):
        img = gr.Image(sources=["upload", "webcam"], label="Screen capture or webcam frame")
        vbtn = gr.Button("Judge this frame", variant="primary")
        vban = gr.Markdown(); vjson = gr.Code(language="json", label="verdict")
        vbtn.click(ui_verdict, [img, events], [vjson, vban, events])
    with gr.Tab("2 · Work-diff — the phone answer"):
        p = gr.Textbox(label="Notes — before", lines=8, value=v1)
        c = gr.Textbox(label="Notes — now", lines=8, value=v2)
        dbtn = gr.Button("Judge my progress", variant="primary")
        djson = gr.Code(language="json"); dbtn.click(ui_diff, [p, c, events], [djson, events])
    with gr.Tab("3 · The Bouncer — earn a break"):
        n = gr.Textbox(label="Your notes", lines=6, value=v2)
        qbtn = gr.Button("Ask me one question"); qstore = gr.Textbox(visible=False); qshow = gr.Markdown()
        a = gr.Textbox(label="Your answer"); gbtn = gr.Button("Submit", variant="primary"); gout = gr.Markdown()
        qbtn.click(ui_bouncer_q, n, [qstore, qshow]); gbtn.click(ui_bouncer_grade, [qstore, a, events], [gout, events])
    with gr.Tab("4 · End session — receipt"):
        rbtn = gr.Button("Generate my receipt", variant="primary")
        rmd = gr.Markdown(); rlm = gr.Code(language="json", label="updated learner model")
        rbtn.click(ui_receipt, events, [rmd, rlm])
demo.launch(share=True)

## Submission checklist (lock 17:00 — submit by 16:40)
- [ ] `MOCK == False` in the final Save Version (cell 1 prints it; eval refuses to be quoted in mock)
- [ ] Real captures added for the `_TODO_*` slots (esp. **phone_in_hand**) → re-run eval → quote the printed number
- [ ] Writeup: track **Motivation & Habits**, ≤1,500 words, headings mirror the rubric
- [ ] This notebook **public** + attached under Project Links (it is the code repo)
- [ ] Test-set dataset **public** + attached as input
- [ ] Gradio share URL in attachments, worded: "live app: <url> — or Run All this notebook"
- [ ] **Save Version (Run All) started by 16:10** (it re-runs everything incl. model download)
- [ ] Keep the interactive session alive through judging (share link dies with it)